# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click !here goes the icon of the corresponding button in the gutter! button.
To debug a cell, press Alt+Shift+Enter, or click !here goes the icon of the corresponding button in the gutter! button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/jupyter-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import io
import random
import os

def get_random_word(fname):
  f = io.open(fname, mode="r", encoding="utf-8")
  all_words = f.read().split('\n')
  return all_words[random.randint(0, len(all_words) - 1)]


def get_greeting():
	greeting_fname = "message_data\\greetings.txt"
	return get_random_word(greeting_fname)


def get_insult():
	insult_fname = "message_data\\insults.txt"
	return get_random_word(insult_fname)


def main():
	print(get_insult())
	print(get_greeting())


In [2]:
from io import BytesIO
from PIL import Image



def get_image_from_bytes(b):
     stream = BytesIO(b)
     image = Image.open(stream).convert("RGB")
     stream.close()
     return image


def nuar_filter(image):
     pix = image.load()
     min_average, max_average = 255, 0
     for x in range(0, image.width):
          for y in range(0, image.height):
               average = int(sum(pix[x, y]) / 3)
               min_average = min(min_average, average)
               max_average = max(max_average, average)
               pix[x, y] = (average, average, average)
     if min_average == max_average:
          max_average += 1
     for x in range(0, image.width):
          for y in range(0, image.height):
               color = int((pix[x, y][0] - min_average) * 255 / (max_average - min_average))
               pix[x, y] = (color, color, color, 255)
     return image


def convert_to_96x96(image):

	return image.resize((96, 96))

In [3]:
    from PIL import Image
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor
from torch.utils.data.dataloader import DataLoader
import torch.nn as nn
import torch.nn.functional as F


def accuracy(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))


class ImageClassificationBase(nn.Module):
    def training_step(self, batch):
        images, labels = batch
        out = self(images)                  
        loss = F.cross_entropy(out, labels) 
        return loss

    def validation_step(self, batch):
        images, labels = batch
        out = self(images)                    
        loss = F.cross_entropy(out, labels)   
        acc = accuracy(out, labels)          
        return {'val_loss': loss.detach(), 'val_acc': acc}

    def validation_epoch_end(self, outputs):
        batch_losses = [x['val_loss'] for x in outputs]
        epoch_loss = torch.stack(batch_losses).mean()   
        batch_accs = [x['val_acc'] for x in outputs]
        epoch_acc = torch.stack(batch_accs).mean()     
        return {'val_loss': epoch_loss.item(), 'val_acc': epoch_acc.item()}

    def epoch_end(self, epoch, result):
        print("Epoch [{}], train_loss: {:.4f}, val_loss: {:.4f}, val_acc: {:.4f}".format(
            epoch, result['train_loss'], result['val_loss'], result['val_acc']))

class Cifar10CnnModel(ImageClassificationBase):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 93, kernel_size=4, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 256 x 48 x 48

            nn.Conv2d(93, 256, kernel_size=4, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 400, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 384 x 24 x 24

            nn.Conv2d(400, 400, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(400, 256, kernel_size=2, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 256 x 12 x 12

            nn.Flatten(),
            nn.Linear(256 * 12 * 12, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 8))

    def forward(self, xb):
        return self.network(xb)


IndentationError: expected an indented block after function definition on line 11 (163675622.py, line 12)

In [69]:

#os.chdir("models")
model = None # torch.load('facialexpression2.pth')


def get_default_device():
    """Pick GPU if available, else CPU"""
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')


device = get_default_device()


def to_device(data, device):
    """Move tensor(s) to chosen device"""
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)


def predict_image(img, model):
    list = ["вы злой))", "фууу!(отвращение)", "ах-ах-ах, испугался", "кто у нас такой счастиливый?", "томас шелби", "кто-то в тильте...", "вот это уливление!"]

    xb = to_device(img.unsqueeze(0), device)

    yb = model(xb)

    _, preds  = torch.max(yb, dim=1)

    return list[preds[0].item()]


def load_model():
    model = torch.load('models\\facialexpression2.pth')
    print('loaded')


def get_result(image):
	tr = transforms.ToTensor()
	tensor = tr(image)
	return predict_image(tensor, model)

IndentationError: expected an indented block after function definition on line 11 (750843192.py, line 12)

In [50]:
# @emotion_qualifier_bot

import telebot
from io import BytesIO
from PIL import Image


TOKEN = 'your_token'

bot = telebot.TeleBot(TOKEN)


@bot.message_handler(commands=['start'])
def start(message):
    print(f"{message.from_user.first_name} started a bot")
    bot.send_message(message.chat.id, text=get_greeting())


@bot.message_handler(content_types=['text'])
def get_text_messages(message):
    print(f"{message.from_user.first_name} sent message: {message.text}")

    answer = get_insult() + '\nотправьте изображение'
    bot.send_message(message.chat.id, answer)
    print(f"{message.from_user.first_name} received message: {answer}")


@bot.message_handler(content_types=["photo"])
def get_text_messages(message):
    print(f"{message.from_user.first_name} sended an image")

    photo_id = message.photo[-1].file_id
    photo_file = bot.get_file(photo_id)
    photo_bytes = bot.download_file(photo_file.file_path)

    image = get_image_from_bytes(photo_bytes)
    image = convert_to_96x96(image)
    # image = nuar_filter(image)

    # convert image to result
    # then send result to user

    result = get_result(image)
    print(f"{message.from_user.first_name} got result: {result}")

    bot.send_photo(message.chat.id, image)
    bot.send_message(message.chat.id, result)


def start_bot():
    print('bot started')
    load_model()
    bot.infinity_polling(none_stop = True)
    print('bot ended')


In [51]:
def main():
	start_bot()

In [4]:
main()

FileNotFoundError: [Errno 2] No such file or directory: 'message_data\\insults.txt'